# AoT experiment — data exploration

This notebook loads one participant's data and walks through the per-phase shape, confidence ratings, RT distributions, forward bias, and catch-trial pass rate.

**Setup**:
1. `pip install -r analysis/requirements.txt`
2. Click *Download saved data* on the local-dev end page.
3. Drop the resulting `aot_data_*.json` (or a CSV from DataPipe) into `analysis/data/`.
4. Set `DATA_FILE` below and run all cells.

**What's here**:
- Data overview by phase / trial_type_tag
- Layer A familiarization summary
- Main task: RT, confidence, forward-response bias, missed trials
- Catch-trial pass rate (joined with the private manifest)
- Confidence × direction crosstab (calibration is one PR away — see notes)

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# So we can import load_data.py from this notebook regardless of CWD.
sys.path.insert(0, str(Path.cwd()))
from load_data import load_session, list_sessions, load_private_manifest, join_with_private

sns.set_theme(style='whitegrid')
pd.options.display.max_columns = 60
pd.options.display.width = 200

## Pick a session

Local dev mode keeps every session you've ever run in the same browser localStorage. The "Download saved data" button bundles all of them, so a single export usually contains multiple sessions. The cell below lists every session it can find and (by default) loads the most recent one. Set `SESSION_PID` below to one of the listed pids if you want a different one.

In [ ]:
# Pick the most recent JSON bundle in analysis/data/ as a default;
# override with an explicit path if you want a specific one.
_data_dir = Path('data')
_candidates = sorted(
    list(_data_dir.glob('aot_data_*.json')) + list(_data_dir.glob('*.csv')),
    key=lambda p: p.stat().st_mtime,
)
DATA_FILE = _candidates[-1] if _candidates else None
assert DATA_FILE, f"no data files found under {_data_dir.resolve()} — drop one in and re-run."
print('data file:', DATA_FILE)
print()

# Local-dev exports keep every session you've ever run on this machine
# in the same JSON bundle (localStorage accumulates). List them here so
# you can see which one you want and disambiguate. Sessions saved with
# session_start_ms (post-fix) are sorted most-recent-first; legacy
# sessions without a timestamp sink to the bottom but are still listed.
sessions = list_sessions(DATA_FILE)
if len(sessions):
    print('sessions in this bundle:')
    print(sessions.to_string(index=False))
print()

# To pick a specific session, set SESSION_PID to one of the pids above
# and re-run from this cell. Default (None) = most recent session that
# has a session_start_ms; falls back to the last `_final` in iteration
# order for legacy data without a timestamp.
SESSION_PID = None

df = load_session(DATA_FILE, session_pid=SESSION_PID)
print(f'loaded {len(df):,} rows  (key: {df.attrs.get("chosen_key")})')

## Data overview

Row counts by phase and trial_type_tag. The canonical "what did the participant do?" rows are tagged `stimulus`; everything else is auxiliary (start prompts, video plays, ITIs, feedback screens, instruction prompts, save markers).

In [ ]:
if 'phase' in df.columns:
    print('rows by phase:')
    print(df['phase'].value_counts(dropna=False).to_string())
    print()
if 'trial_type_tag' in df.columns:
    print('rows by trial_type_tag:')
    print(df['trial_type_tag'].value_counts(dropna=False).to_string())

## Layer A — familiarization

8 trials in three flavours (direction-only, confidence-only, combined). Rows tagged `phase=familiarization` and `trial_type_tag=stimulus` are the gating rows used by the consecutive-fail check.

In [ ]:
fam = df.query("phase == 'familiarization' and trial_type_tag == 'stimulus'")
if len(fam):
    print(f'familiarization stimulus rows: {len(fam)}')
    if 'correct' in fam.columns:
        print(f'  correct: {(fam["correct"] == True).sum()} / {len(fam)} '
              f'({(fam["correct"] == True).mean():.0%})')
    if 'direction_rt' in fam.columns and fam['direction_rt'].notna().any():
        print(f'  direction_rt: median={fam["direction_rt"].median():.0f} ms')
    if 'confidence_rt' in fam.columns and fam['confidence_rt'].notna().any():
        print(f'  confidence_rt: median={fam["confidence_rt"].median():.0f} ms')
else:
    print('no familiarization rows found.')

## Main task — direction RT distribution

Direction RT is measured from the moment the response prompt appears (= the moment responses become enabled). With the configured 2-s window, RTs > 2000 ms can't occur — a missing direction response shows as NaN.

In [ ]:
main = df.query("phase == 'main' and trial_type_tag == 'stimulus'").copy()
n_main = len(main)
n_missed = main['direction_rt'].isna().sum() if 'direction_rt' in main.columns else 0
print(f'main-task stimulus rows: {n_main}  (missed direction: {n_missed})')

if n_main and 'direction_rt' in main.columns:
    fig, ax = plt.subplots(figsize=(7, 3.2))
    sns.histplot(main['direction_rt'].dropna(), bins=30, kde=True, ax=ax, color='steelblue')
    ax.set_xlabel('direction RT (ms, from prompt onset)')
    ax.set_ylabel('count')
    ax.set_title('Main-task direction RT')
    plt.tight_layout(); plt.show()
    print(f'  median: {main["direction_rt"].median():.0f} ms,  '
          f'mean: {main["direction_rt"].mean():.0f} ms,  '
          f'min/max: {main["direction_rt"].min():.0f}/{main["direction_rt"].max():.0f}')

## Confidence rating distribution

Self-reported confidence (1–5) on main-task rows. A flat distribution can hint at participants picking a fixed value rather than calibrating; a strong peak at 5 means the task feels too easy at this resolution.

In [ ]:
if 'confidence' in main.columns and main['confidence'].notna().any():
    fig, ax = plt.subplots(figsize=(6, 3))
    counts = main['confidence'].value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)
    counts.plot.bar(ax=ax, color='cadetblue')
    ax.set_xlabel('confidence (1=guess, 5=certain)')
    ax.set_ylabel('count')
    ax.set_title('Main-task confidence ratings')
    plt.tight_layout(); plt.show()
    print(counts.to_string())
else:
    print('no confidence ratings on main-task rows.')

## Forward-response bias

We don't have ground truth on the client for main trials, so we can't compute accuracy yet. But we *can* compute the rate at which the participant pressed FORWARD vs BACKWARD. With balanced 50/50 stimuli, a robust forward bias is the expected human signature (Meding et al.: ~30% errors on reversed clips vs ~9% on forward → forward responses ≫ 50%).

In [ ]:
if 'response_direction' in main.columns:
    rd = main['response_direction'].value_counts(dropna=False)
    print(rd.to_string())
    answered = main['response_direction'].dropna()
    if len(answered):
        forward_rate = (answered == 'forward').mean()
        print(f'\nforward-response rate (over {len(answered)} answered): {forward_rate:.1%}')

## Per-block summary

In [ ]:
if 'block_index' in main.columns:
    g = main.groupby('block_index')
    summary = pd.DataFrame({
        'n_trials':     g.size(),
        'n_missed':     g['direction_rt'].apply(lambda s: s.isna().sum()),
        'median_rt_ms': g['direction_rt'].median().round(0),
        'forward_rate': g['response_direction'].apply(
            lambda s: (s == 'forward').mean()),
        'mean_conf':    g['confidence'].mean().round(2) if 'confidence' in main.columns else np.nan,
    })
    print(summary.to_string())

## Catch-trial pass rate (offline scoring)

We need the private manifest to identify which trials were catches and what the expected response was. The private manifest is gitignored — set `PRIVATE_MANIFEST` to its real path on this machine.

In [ ]:
PRIVATE_MANIFEST = Path('../secrets/manifest_private.json')
if not PRIVATE_MANIFEST.exists():
    print(f'note: {PRIVATE_MANIFEST} not found — skipping offline scoring.')
    private = None
else:
    private = load_private_manifest(PRIVATE_MANIFEST)
    print(f'private manifest: {len(private):,} entries')
    print(private['type'].value_counts().to_string())

In [ ]:
if private is not None:
    df_joined = join_with_private(df, private)
    main_joined = df_joined.query("phase == 'main' and trial_type_tag == 'stimulus'")

    # Catch-trial encounters and pass rate.
    catches = main_joined[main_joined['pm_type'] == 'catch']
    n_catch = len(catches)
    n_pass = catches['catch_passed'].fillna(False).sum() if 'catch_passed' in catches else 0
    print(f'catch trials encountered: {n_catch}')
    if n_catch:
        print(f'  passed (direction + confidence both correct): {n_pass} / {n_catch} = {n_pass/n_catch:.0%}')
        print()
        cols = [c for c in [
            'block_index', 'trial_index_in_block',
            'pm_direction', 'pm_expected_confidence',
            'response_direction', 'confidence', 'catch_passed',
        ] if c in catches.columns]
        print(catches[cols].to_string())

    # Real-clip accuracy and forward-bias error breakdown.
    real = main_joined[main_joined['pm_type'] == 'main']
    if len(real):
        answered = real.dropna(subset=['response_direction'])
        acc = answered['main_correct'].mean()
        print(f'\nmain-task direction accuracy: {acc:.1%}  ({len(answered)} answered)')
        # Errors broken down by stimulus direction (the Meding signature)
        err_fwd = answered.query("pm_direction == 'forward'")
        err_rv  = answered.query("pm_direction == 'backward'")
        if len(err_fwd) and len(err_rv):
            print(f'  errors on forward stimuli:  {(~err_fwd["main_correct"]).mean():.1%}  '
                  f'({len(err_fwd)} trials)')
            print(f'  errors on backward stimuli: {(~err_rv["main_correct"]).mean():.1%}  '
                  f'({len(err_rv)} trials)')

## Confidence calibration (offline)

Accuracy as a function of self-reported confidence. A well-calibrated participant is monotonically increasing here. A flat line is a red flag (the participant isn't tracking their own uncertainty).

In [ ]:
if private is not None and 'main_joined' in dir():
    real = main_joined[main_joined['pm_type'] == 'main'].dropna(subset=['confidence', 'main_correct'])
    if len(real):
        cal = real.groupby('confidence').agg(
            n=('main_correct', 'size'),
            accuracy=('main_correct', 'mean'),
        )
        print(cal.to_string())
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.plot(cal.index, cal['accuracy'], marker='o')
        ax.axhline(0.5, color='grey', linestyle='--', alpha=0.5)
        ax.set_xlim(0.5, 5.5)
        ax.set_ylim(0, 1)
        ax.set_xlabel('self-reported confidence')
        ax.set_ylabel('direction accuracy')
        ax.set_title('Confidence calibration')
        plt.tight_layout(); plt.show()

## Full-corpus pipeline

Once you have downloaded sessions into `analysis/data/` (JATOS multi-file
exports under `study_result_*/comp-result_*/files/`, or single JSON/CSV
files), run the four-stage pipeline below. Each stage caches its output
under `analysis/derived/` (gitignored), so you only re-run downstream
stages when their inputs change.


In [ ]:
from analysis.load_all import build_responses_db
from analysis.per_subject import build_per_subject_table
from analysis.per_video import build_per_video_table
from analysis.per_source import build_per_source_table

responses = build_responses_db()           # → analysis/derived/responses.parquet
per_subj  = build_per_subject_table()      # → per_subject.tsv + per-subject PNGs
per_vid   = build_per_video_table()        # → per_video.tsv (uses subject_quality_weight)
per_src   = build_per_source_table()       # → per_source.tsv (forward vs backward pivot)


### Per-subject inspection

`per_subject.tsv` is the ranking table you'd use to decide who to include in further
analyses. Headline columns:

- `included` — boolean, hard inclusion gate from CLAUDE.md §3.9
- `subject_quality_weight ∈ [0, 1]` — continuous weight used in per_video aggregation
- `d_prime` / `meta_d_prime` / `m_ratio` — Type-1 ability + Type-2 metacognition
- `metacog_source` — `m_ratio` when meta-d' converged, `calib_gamma_fallback` otherwise
- `catch_full_pass_rate` — attention probe (≥ 0.80 is the inclusion threshold)
- `forward_bias` — Hanyu et al. comparison (large + bias is the published signature)

The per-subject PNG (under `analysis/derived/figures/per_subject/<pid_hash>.png`)
gives a one-page visual sanity check per session.


In [ ]:
import pandas as pd
ps = pd.read_csv('analysis/derived/per_subject.tsv', sep='\t')
ps[['pid_hash', 'included', 'subject_quality_weight', 'd_prime',
    'meta_d_prime', 'm_ratio', 'catch_full_pass_rate',
    'main_direction_accuracy', 'forward_bias']]


### Per-video identifiability — the headline ranking

`identifiability_score ∈ [−1, +1]` is the confidence-weighted "bettor" score:

- **+1.0** — every viewer confidently and correctly identified the clip's direction.
- **0.0** — chance / split crowd / everyone guessed.
- **−1.0** — every viewer confidently *wrong*. These are the most interesting clips — visually they suggest the opposite direction even though they're forward-going.

`direction_accuracy_raw` is the equal-weighted companion (ignores confidence). When the two disagree, the clip is doing something interesting.


In [ ]:
pv = pd.read_csv('analysis/derived/per_video.tsv', sep='\t')
print(f'{len(pv)} (stim × direction) cells, n_views distribution: {pv["n_views"].value_counts().to_dict()}')
print()
print('Most identifiable forward clips:')
fwd_top = pv[pv['pm_direction'] == 'forward'].head(10)[['stimulus_id', 'n_views', 'identifiability_score', 'direction_accuracy_raw']]
print(fwd_top.to_string(index=False))
print()
print('Least identifiable (anti-id, where everyone is confidently wrong):')
print(pv.dropna(subset=['identifiability_score']).tail(10)[['stimulus_id', 'pm_direction', 'n_views', 'identifiability_score', 'direction_accuracy_raw', 'mean_confidence_wrong']].to_string(index=False))


### Per-source asymmetry

`per_source.tsv` pivots forward and reverse renders of the same source video into one row. `asymmetry = id_score_fw − id_score_bw`. Large `|asymmetry|` flags clips where one direction is much harder than the other — interesting both for the AoT-cue literature and for stimulus design (these are the clips that *teach* you which direction is forward in their domain).


In [ ]:
psrc = pd.read_csv('analysis/derived/per_source.tsv', sep='\t')
psrc.head(15)[['source_id', 'identifiability_score_fw', 'identifiability_score_bw',
               'asymmetry', 'mean_identifiability', 'preferred_direction',
               'n_views_fw', 'n_views_bw']]
